### Implementing RAG using langchain

## Installing and importing Libraries for Data Ingestion (Step 1) in RAG

In [2]:
!pip install uv

In [3]:
!uv pip install langchain langchain-core langchain-community
!uv pip install pypdf pymupdf "transformers==4.38.0" "sentence-transformers==2.5.1" chromadb

Using Python 3.12.13 environment at: /usr
Checked 3 packages in 124ms
Using Python 3.12.13 environment at: /usr
Checked 5 packages in 136ms


## Langchain Document and Document loader class

In [4]:
from langchain_core.documents import Document

In [5]:
sample_doc = Document(
        page_content="Hello, world!", metadata={"source": "https://example.com"}
)

In [6]:
sample_doc

Document(metadata={'source': 'https://example.com'}, page_content='Hello, world!')

In [7]:
type(sample_doc)

langchain_core.documents.base.Document

### Loading txt file data into Document
#### raw data (txt format) into Document [raw data ==> Document]

Reference : https://reference.langchain.com/python/langchain-core/document-loaders

In [8]:
## Text Data
from langchain_community.document_loaders.text import TextLoader

loader = TextLoader("403930086-Astanga-hrdayam-Eng-pdf.txt",encoding = "utf-8")

In [9]:
document = loader.load()

In [10]:
document

[Document(metadata={'source': '403930086-Astanga-hrdayam-Eng-pdf.txt'}, page_content='           ILLUSTRATED\n\n\nTEXT WITH ENGLISH TRANSLATION AND APPENDICES\n\x0c              The\n   CHAUKHAMBA AYURVEDA STUDIES\n                     15\n\n\n\n\n               Illustrated\n Astanga Hrdaya\n           of Vagbhata\n\n      SUTRA-STHANA\n        Text with English Translation\n\n\n                     #\n                 including\n       MAULIKA SIDDHANTA\n       [as per CCIM Syllabus 2012]\n\n                      by\n           Dr. R. VIDYANATH\n                 MD (Ayu); PhD\n                Professor & HOD\n        P G Dept. of Ayurveda Samhita\n      Dr. B R K R Govt. Ayurvedic College\n           HYDERABAD-500038 (A.P.)\n\n                 foreword by\n            Prof. R.H. SINGH\n\n\n\n\nChaukhamba Surbharati Prakashan\n                  Varanasi\n\x0c© All right reserved. No part of this publication may be reproduced or transmitted in any form or by any\nmeans, electronic or me

In [11]:
## PDF Data
from langchain_community.document_loaders.pdf import PyPDFLoader

pdf_loader = PyPDFLoader("pdf_Folder/NIPS-2017-attention-is-all-you-need-Paper.pdf")
pdf_document = pdf_loader.load()

pdf_document

[Document(metadata={'producer': 'pdfcpu v0.9.1 dev', 'creator': 'PyPDF', 'creationdate': '2026-04-10T16:47:24+00:00', 'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin', 'book': 'Advances in Neural Information Processing Systems 30', 'created': '2017', 'date': '2017', 'description': 'Paper accepted and presented at the Neural Information Processing Systems Conference (http://nips.cc/)', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more paralleli

## Ingestion Pipeline

In [12]:
## Data ==> Documents
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

In [13]:
def load_all_pdfs():
  folder_path = "pdf_Folder"
  num_docs = 0
  all_docs = []

  for filename in os.listdir(folder_path):
    if filename.lower().endswith(".pdf"):
      ## complete file path
      pdf_path = os.path.join(folder_path,filename)

      loader = PyPDFLoader(pdf_path)
      doc = loader.load()

      all_docs.extend(doc)
      num_docs += 1

  print("Total pdfs:", num_docs)
  print("Total pages:", len(all_docs))
  return all_docs

In [14]:
all_pdf_doc = load_all_pdfs()

Total pdfs: 3
Total pages: 588


In [15]:
type(all_pdf_doc)

list

## (Step 2) Chunking Documents
raw data ==> Documents ==> Chunking/Text splitting

Reference: https://reference.langchain.com/python/langchain-text-splitters/character/RecursiveCharacterTextSplitter

In [16]:
### Chunking
!uv pip install langchain-text-splitters

Using Python 3.12.13 environment at: /usr
Checked 1 package in 76ms


In [17]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_doc_into_chunk(documents, chunk_size=500,chunk_overlap=50):
  text_splitter = RecursiveCharacterTextSplitter(
      chunk_size = chunk_size,
      chunk_overlap = chunk_overlap
      )

  chunked_document = text_splitter.split_documents(documents)
  return chunked_document

In [18]:
chunks = split_doc_into_chunk(all_pdf_doc)

In [19]:
len(chunks)

218

In [20]:
chunks

[Document(metadata={'producer': 'pdfcpu v0.9.1 dev', 'creator': 'PyPDF', 'creationdate': '2026-04-10T16:47:24+00:00', 'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin', 'book': 'Advances in Neural Information Processing Systems 30', 'created': '2017', 'date': '2017', 'description': 'Paper accepted and presented at the Neural Information Processing Systems Conference (http://nips.cc/)', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more paralleli

## (Step 3) Embeddings
raw data ==> Documents ==> Chunking ==> Embeddings

In [21]:
from sentence_transformers import SentenceTransformer

class EmbeddingManager:

  def __init__(self,model_name="all-MiniLM-L6-v2"):
    self.model_name = model_name
    self.model = SentenceTransformer(model_name)
    print("Embedding dimensions= ", self.model.get_sentence_embedding_dimension())

  def generate_embeddings(self, text):
    embeddings = self.model.encode(text,show_progress_bar=True)
    print(embeddings.shape)
    return embeddings

In [22]:
# Run the installation cell (WmL77HkwTtuR) first, then restart the runtime using the cell I added below it, and finally run this:
embedding_manager = EmbeddingManager()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dimensions=  384


## (Step 4) Vector Store or Vector Database
raw data ==> Documents ==> Chunking ==> Embeddings ==> Vector Store/Vector Database



In [23]:
import chromadb
import uuid

In [24]:
class Vector_Store_Manager:
  def __init__(self,persist_directory="vector_store",collection_name="pdf_documents"):
    self.collection_name = collection_name
    self.persist_directory = persist_directory
    self.collection = None
    self.client = None

    self._initialize_store()

  def _initialize_store(self):
    if not os.path.exists(self.persist_directory):
      os.makedirs(self.persist_directory,exist_ok=True)
      print("Creating new vector store")

    #Create a client
    self.client = chromadb.PersistentClient(path=self.persist_directory)

    #Create the collection
    self.collection = self.client.get_or_create_collection(
        name = self.collection_name,
        metadata={"description": "vector store collection for pdf embeddings in RAG"}
    )

    print("Initialized the vector store with collection",self.collection_name)
    print("docs in collection: ", self.collection.count())

  def add_documents(self,documents,embeddings):
    if len(documents) != len(embeddings):
      raise ValueError("Number of documents and embeddings must be equal")


    # Store => ids, embedding, document, metadata
    ids,all_metadata,documents_content, embeddings_list = [],[],[],[]

    for i, (doc, embedding) in enumerate(zip(documents,embeddings)):
      doc_id = f"doc_{uuid.uuid4()}"
      ids.append(doc_id)

      metadata = dict(doc.metadata)
      metadata["doc_index"] = 1
      metadata["content_length"] = len(doc.page_content)
      all_metadata.append(metadata)

      documents_content.append(doc.page_content)
      embeddings_list.append(embedding.tolist())

    # Move the add call outside the loop for efficiency, or keep inside if needed
    self.collection.add(
        ids=ids,
        metadatas=all_metadata,
        documents=documents_content,
        embeddings=embeddings_list
    )

    # Indented these inside the method so they can access the variables
    print("Total docs added in vector store=",len(documents_content))
    print("docs in collection: ", self.collection.count())

In [25]:
vector_store = Vector_Store_Manager()

Creating new vector store
Initialized the vector store with collection pdf_documents
docs in collection:  0


In [26]:
## data ==> documents ==> chunks ==> embeddings ==> Store in vector store

texts = [doc.page_content for doc in chunks]
embedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks,embedding)

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

(218, 384)
Total docs added in vector store= 218
docs in collection:  218


## Retrieval Pipeline

In [29]:
## Using cosine similarity algorithm
from sklearn.metrics.pairwise import cosine_similarity

In [43]:
class RagRetriever:
  def __init__(self,embedding_manager,vector_store):
    self.embedding_manager = embedding_manager
    self.vector_store = vector_store

  def retrieve(self,query,top_k=5,score_threshold=0.0):
    # Convert query into embeddings
    query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

    #Retrieve data from Vector store
    ## Semantic search
    results = self.vector_store.collection.query(
        query_embeddings = [query_embeddings.tolist()],
        n_results = top_k
    )

    # Cosine similarity
    retrieved_docs = []
    if results["documents"] and results["documents"][0]:
      ids = results["ids"][0]
      metadatas = results["metadatas"][0]
      documents = results["documents"][0]
      distances = results["distances"][0]

      for i, (doc_id,metadata,document,distance) in enumerate(zip(ids,metadatas,documents,distances)):
        similarity_score = 1 - distance

        if similarity_score >= score_threshold:
          retrieved_docs.append({
               "id": doc_id,
               "document": document,
               "metadata": metadata,
               "distance": distance,
               "similarity_score": similarity_score,
               "rank": i + 1
              })

      print(f"retrieved {len(retrieved_docs)} documents")

    else:
      print("No documents found")


    return retrieved_docs

In [45]:
ragRetriever = RagRetriever(embedding_manager,vector_store)

In [50]:
ragRetriever.retrieve("What is Attention ?")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

(1, 384)
retrieved 5 documents


[{'id': 'doc_213aed72-2aae-4305-9e77-3fb61471d2b4',
  'document': 'sub-layer in the decoder stack to prevent positions from attending to subsequent positions. This\nmasking, combined with fact that the output embeddings are offset by one position, ensures that the\npredictions for positioni can depend only on the known outputs at positions less thani.\n3.2 Attention\nAn attention function can be described as mapping a query and a set of key-value pairs to an output,\nwhere the query, keys, values, and output are all vectors. The output is computed as a weighted sum',
  'metadata': {'editors': 'I. Guyon and U.V. Luxburg and S. Bengio and H. Wallach and R. Fergus and S. Vishwanathan and R. Garnett',
   'subject': 'Neural Information Processing Systems http://nips.cc/',
   'language': 'en-US',
   'producer': 'pdfcpu v0.9.1 dev',
   'publisher': 'Curran Associates, Inc.',
   'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional ne

In [51]:
ragRetriever.retrieve("What is encoder decoder ?")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

(1, 384)
retrieved 5 documents


[{'id': 'doc_c5a3571d-1231-4808-b40c-4188a4779688',
  'document': 'layers, produce outputs of dimensiondmodel = 512.\nDecoder: The decoder is also composed of a stack ofN = 6 identical layers. In addition to the two\nsub-layers in each encoder layer, the decoder inserts a third sub-layer, which performs multi-head\nattention over the output of the encoder stack. Similar to the encoder, we employ residual connections\naround each of the sub-layers, followed by layer normalization. We also modify the self-attention',
  'metadata': {'doc_index': 1,
   'producer': 'pdfcpu v0.9.1 dev',
   'published': '2017',
   'creationdate': '2026-04-10T16:47:24+00:00',
   'title': 'Attention is All you Need',
   'content_length': 447,
   'page_label': '3',
   'moddate': '2026-04-10T16:47:24+00:00',
   'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models al

In [59]:
ragRetriever.retrieve("AI powered scam detection system")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

(1, 384)
retrieved 0 documents


[]

# Integrate with LLMs
## Groq

In [65]:
GROQ_API_KEY = "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"

In [66]:
!uv pip install langchain-groq

Using Python 3.12.13 environment at: /usr
Checked 1 package in 79ms


In [71]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key=GROQ_API_KEY,
    model="groq/compound",
    temperature=0.1,
    max_tokens=1024,
    # other params...
)

In [72]:
## generate our RAG output
def generate_output(query,ragRetriever,llm,top_k=5):
  results = ragRetriever.retrieve(query,top_k)

  context = "\n".join(doc["document"] for doc in results) if results else ""

  if not context:
    print("we found no relevant context for the given query")

  # prompt = context + query
  prompt = f""" use given context to generate the answer
                Context : {context}
                Query   : {query} """

  response = llm.invoke([prompt.format(context=context,query=query)])
  return response.content

In [75]:
output = generate_output("what is Attention?", ragRetriever,llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

(1, 384)
retrieved 5 documents


In [76]:
print(output)

**Answer – What is “Attention” (according to the supplied context)**  

---

### Core definition  
- **Attention** is an *attention function* that **maps a query vector together with a set of key‑value pairs to an output vector**.  
- The **query, keys, values, and the resulting output are all vectors**.  
- The output is obtained as a **weighted sum of the value vectors**, where the weights are computed from the similarity between the query and each key (the exact similarity measure is not detailed in the excerpt, but the standard formulation uses a soft‑max over dot‑products).

### Self‑attention (intra‑attention)  
- **Self‑attention** is a special case of attention applied **within a single sequence**: each position of the sequence acts as a query, key, and value simultaneously.  
- By relating *different positions* of the same sequence, self‑attention produces a **context‑aware representation of each token**.  
- It has been successfully used for many NLP tasks (reading comprehens